# sptc on SGLang: Kaggle T4 check of the SGLang scheduler plugin

Runs the SGLang version of the speculation-aware scheduler (`integrations/sptc_sglang`) on
**SGLang 0.5.22, Qwen3-1.7B (fp16), one T4**. Takes about 1 hour.

**Use a new notebook / fresh session** (not the vLLM one): SGLang pins its own torch.

**Settings:** Accelerator **GPU T4**, Internet **on**. **Input:** the dataset `sptc-bundle`
(rebuilt with `kaggle/make_bundle.sh`, so it contains `sptc_sglang`).

**What it does:**
1. Install SGLang and the plugin; run the plugin's tests (local + real-SGLang wiring).
2. Smoke test S1–S4 through the OpenAI API: an idle bet completes, a bet with no spare
   capacity is refused, an `early` call completes (and how much it stalls a normal stream),
   a running bet is dropped under KV pressure.
3. Load test on **stock SGLang**: spec-ptc vs no speculation, with and without one bad guess
   per speculation (same as the vLLM run).
4. The same load test **with the plugin**, decoys tagged `bet`, speculations `early`.

Download `/kaggle/working/sptc_sglang_out.zip` at the end.

In [ ]:
import os, subprocess, sys, time, json, glob, shutil, zipfile

REPO = ""   # optional GitHub URL; leave empty to use the uploaded dataset

def find_bundle(top="/kaggle/input"):
    # a folder that contains both sptc-sim and spec-ptc, at any depth
    for root, dirs, files in os.walk(top):
        if "sptc-sim" in dirs and "spec-ptc" in dirs:
            return root
    # zip uploaded but not unpacked: unpack it into /kaggle/working and search again
    for z in glob.glob(f"{top}/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            f.extractall("/kaggle/working/_unzipped")
        return find_bundle("/kaggle/working/_unzipped")
    return None

if REPO:
    B = "/kaggle/working/sptc-bundle"
    subprocess.run(["git", "clone", "--depth", "1", REPO, f"{B}/sptc-sim"], check=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/alexzhang13/spec-ptc",
                    f"{B}/spec-ptc"], check=True)
else:
    src = find_bundle()
    if src is None:
        print("Bundle not found. Here is what is under /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count(os.sep) <= 6:
                print(root, dirs[:8], files[:5])
        raise SystemExit("bundle not found: see the listing above")
    print("found bundle at", src)
    B = "/kaggle/working/sptc-bundle"
    shutil.copytree(src, B, dirs_exist_ok=True)  # inputs are read-only

SIM, SPTC = f"{B}/sptc-sim", f"{B}/spec-ptc"
OUT = "/kaggle/working/sptc_sglang_out"
os.makedirs(OUT, exist_ok=True)
print(SIM, os.listdir(SIM))
print(SPTC, os.listdir(SPTC))
!nvidia-smi

In [ ]:
%%bash -s "$SPTC" "$SIM"
pip -q install "sglang==0.5.22" 2>&1 | tail -3
pip -q install pandas numpy pytest openai 2>&1 | tail -1
pip -q install -e "$1" 2>&1 | tail -1                  # spec-ptc
pip -q install -e "$2/integrations" --no-deps 2>&1 | tail -1   # the plugin (entry point)
python -c "import sglang, torch, spec_ptc; print('sglang', sglang.__version__, 'torch', torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))"
python -c "import importlib.metadata as m; print([e for e in m.entry_points(group='sglang.srt.plugins')])"

In [ ]:
# plugin tests: local (policy, SGLang logic on fakes; the static source checks skip here)
# and the real-SGLang wiring tests
!cd {SIM}/integrations && python -m pytest -q -p no:cacheprovider tests tests_sglang 2>&1 | tail -15

In [ ]:
import subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-1.7B"
# T4 = compute capability 7.5. If SGLang's default kernels refuse it, start_sglang retries
# once with the portable backends and keeps them for every later start.
FALLBACK = ["--attention-backend", "triton", "--sampling-backend", "pytorch"]
EXTRA = []

def _launch(flags, log, env):
    cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", MODEL, "--dtype", "half",
           "--host", "127.0.0.1", "--port", "8000", "--mem-fraction-static", "0.85", *flags]
    return subprocess.Popen(cmd, stdout=open(f"{OUT}/{log}", "w"), stderr=subprocess.STDOUT, env=env)

def start_sglang(extra=(), log="sglang.log", sptc=None):
    """sptc: None = stock SGLang; a dict = plugin on with these knobs."""
    global EXTRA
    env = {k: v for k, v in os.environ.items() if k != "SPTC_SGLANG_CONFIG"}
    if sptc is not None:
        env["SPTC_SGLANG_CONFIG"] = json.dumps(sptc)
    for attempt in range(2):
        p = _launch([*EXTRA, *extra], log, env)
        for _ in range(900):
            try:
                urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
                print("up", "(fallback kernels)" if EXTRA else "")
                txt = open(f"{OUT}/{log}").read()
                if sptc is not None:
                    print("plugin line:", [l for l in txt.splitlines() if "sptc scheduler on" in l][:1] or "MISSING")
                return p
            except Exception:
                if p.poll() is not None:
                    break
                time.sleep(2)
        tail = open(f"{OUT}/{log}").read()[-2500:]
        if p.poll() is None:
            p.terminate()
        if attempt == 0 and not EXTRA:
            print("start failed, retrying with", FALLBACK, "\n---\n", tail[-1200:])
            EXTRA = FALLBACK
            time.sleep(10)
            continue
        raise RuntimeError(tail)

def stop_sglang(p):
    p.terminate()
    try:
        p.wait(timeout=120)
    except subprocess.TimeoutExpired:
        p.kill()
    time.sleep(10)

In [ ]:
# smoke S1-S3 (free_window 2 so two decoding streams leave no room for a bet)
SMOKE = f"python {SIM}/integrations/kaggle/sptc_sglang_smoke.py --model {MODEL}"
srv = start_sglang(["--context-length", "4096"], log="sglang_smoke123.log", sptc={"free_window": 2})
cmd, log = f"{SMOKE} --cases s1,s2,s3", f"{OUT}/smoke_s123.jsonl"
!{cmd} 2>&1 | grep check | tee {log}
stop_sglang(srv)
# S4: a small KV pool (3200 tokens) so 1 bet + 3 requests of ~600 prompt + 1500 output
# tokens overflow it while decoding: SGLang must retract, and the bet goes first (dropped)
srv = start_sglang(["--max-total-tokens", "3200", "--context-length", "2560"],
                   log="sglang_smoke4.log", sptc={})
cmd, log = f"{SMOKE} --cases s4", f"{OUT}/smoke_s4.jsonl"
!{cmd} 2>&1 | grep check | tee {log}
stop_sglang(srv)
!grep -c Traceback {OUT}/sglang_smoke123.log {OUT}/sglang_smoke4.log || true
!grep -h "Retract\|retract" {OUT}/sglang_smoke4.log | head -3

In [ ]:
# load test on STOCK SGLang (no plugin): baseline vs spec-ptc, 0 and 1 bad guess (~30 min)
BENCH = f"python {SIM}/kaggle/vllm_bench.py"
srv = start_sglang(["--context-length", "8192"], log="sglang_stock.log")
!{BENCH} load --engine sglang --model {MODEL} --out {OUT} --spec-ptc {SPTC} --agents 1,4,8 \
    --decoys 0,1 --turns 4 --patterns map16,map_reduce,two_stage,side_branch_chain
stop_sglang(srv)

In [ ]:
# the same load test WITH the plugin, decoys tagged bet, speculations early (~30 min)
srv = start_sglang(["--context-length", "8192"], log="sglang_plugin.log",
                   sptc={"free_window": 8, "kv_reserve": 0.10, "spec_prefill_budget": 512})
!{BENCH} load --engine sglang --model {MODEL} --out {OUT} --spec-ptc {SPTC} --agents 1,4,8 \
    --decoys 0,1 --turns 4 --patterns map16,map_reduce,two_stage,side_branch_chain --tag-bets
stop_sglang(srv)
!grep -c Traceback {OUT}/sglang_stock.log {OUT}/sglang_plugin.log || true

In [ ]:
!cd /kaggle/working && zip -qr sptc_sglang_out.zip sptc_sglang_out && ls -la sptc_sglang_out.zip
for f in ("smoke_s123.jsonl", "smoke_s4.jsonl"):
    p = f"{OUT}/{f}"
    print(f, open(p).read() if os.path.exists(p) else "missing")